## Temprature Data Cleaning 

This notebook processes historical temperature data for major U.S. cities to prepare region-specific datasets for further analysis. The workflow includes:

- **Loading Data:** The full city-level temperature dataset is loaded from `temperatureData/GlobalLandTemperaturesByCity.csv`. Only U.S. cities are kept, and columns are renamed for clarity.
- **Region Mapping:** A dictionary (`region_to_cities`) maps PJM power regions to their associated cities.
- **Data Extraction & Saving:** For each PJM region, the code filters the temperature data to include only the relevant cities. If data exists for the region, it determines the date range and saves the filtered data as a CSV file in the `temperatureData_clean` folder. The filenames include the region name and date range for easy identification.
- **Error Handling:** If no temperature data is found for a region, a warning is printed.

This process results in a set of cleaned, region-specific temperature files ready for use in energy or climate analysis.

In [10]:
# Imports & load full city-level dataset
import pandas as pd
import os

city_path = "temperatureData/GlobalLandTemperaturesByMajorCity.csv"
df_city = pd.read_csv(city_path, parse_dates=["dt"])

# Keep U.S. rows and rename columns
df_city = (
    df_city[df_city["Country"] == "United States"]
      .rename(columns={"dt": "Date",
                       "City": "Location",
                       "AverageTemperature": "Temp_C"})
      [["Date", "Location", "Temp_C"]]
      .dropna(subset=["Temp_C"])
)

In [11]:
# Define cities associated with each PJM region
region_to_cities = {
    "AEP": ["Akron", "Columbus", "Dayton", "Charleston",
            "Indianapolis", "Lexington Fayette", "Huntington"],
    "COMED": ["Chicago", "Rockford", "Aurora", "Naperville", "Joliet"],
    "DAYTON": ["Dayton"],
    "DEOK": ["Cincinnati", "Lexington Fayette"],
    "DOM": ["Charlotte", "Raleigh", "Richmond",
            "Virginia Beach", "Washington"],
    "DUQ": ["Pittsburgh"],
    "EKPC": ["Lexington Fayette", "Louisville"],
    "FE": ["Akron", "Allentown", "Cleveland", "Newark", "Baltimore"],
    "NI": ["South Bend", "Fort Wayne"],
    "PJM_Load": ["Columbus", "Detroit", "Charlotte", "Chicago", "Baltimore",
                 "Philadelphia", "Cleveland", "Indianapolis", "Louisville",
                 "Nashville", "Jersey City", "Akron"],
    "PJME": ["Baltimore", "Newark", "Jersey City", "Washington"],
    "PJMW": ["Charleston", "Lexington Fayette", "Columbus", "Dayton"],
}



In [12]:
# Create output folder
os.makedirs("temperatureData_clean", exist_ok=True)

# Loop through all cleaned energy files
for region, cities in region_to_cities.items():

    region_temp = df_city[df_city["Location"].isin(cities)].copy()

    if region_temp.empty:
        print(f" No temperature rows found for {region}")
        continue

    date_min, date_max = region_temp["Date"].min(), region_temp["Date"].max()
    date_str_min = date_min.strftime("%Y-%m-%d")
    date_str_max = date_max.strftime("%Y-%m-%d")

    # use "PJM" instead of "PJM_Load" in filename
    file_region = "PJM" if region == "PJM_Load" else region
    outfile = f"temperatureData_clean/{file_region}_{date_str_min}_to_{date_str_max}.csv"

    region_temp.to_csv(outfile, index=False)
    print(f" Saved {file_region} temperature data → {outfile}")

 No temperature rows found for AEP
 Saved COMED temperature data → temperatureData_clean/COMED_1743-11-01_to_2013-09-01.csv
 No temperature rows found for DAYTON
 No temperature rows found for DEOK
 No temperature rows found for DOM
 No temperature rows found for DUQ
 No temperature rows found for EKPC
 No temperature rows found for FE
 No temperature rows found for NI
 Saved PJM temperature data → temperatureData_clean/PJM_1743-11-01_to_2013-09-01.csv
 No temperature rows found for PJME
 No temperature rows found for PJMW
